# Model Validation & Ablation Testing
In this notebook, we verify our models' robustness using Cross-Validation and perform an Ablation Test to definitively prove the value of our engineered features.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Load data
df = pd.read_csv('../data/processed.csv')

# Setup data
X_full = df.drop(columns=['isFraud', 'step'])
y = df['isFraud']
print(f"Data loaded. Shape: {X_full.shape}")

Data loaded. Shape: (217575, 9)


## 1. Setting up Models & Cross-Validation
**What is Cross-Validation (CV)?**
Previously, we split the data into 80% train and 20% test *once*. What if we got lucky (or unlucky) with that specific 20% slice? 
In 5-fold cross-validation, we slice the data into 5 equal chunks. We train the model on 4 chunks and test on the 1 remaining chunk. We repeat this 5 times until *every* chunk has been used as the test set once. 

**What does Standard Deviation (Std) tell us?**
We calculate the average (mean) score across all 5 tests. The Standard Deviation tells us how much the scores bounced up and down between the 5 tests. A small standard deviation means the model is highly stable and reliable, no matter what data you throw at it.

In [2]:
# Set up the models with the exact same settings as before
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)

num_non_fraud = (y == 0).sum()
num_fraud = (y == 1).sum()
scale_weight = num_non_fraud / num_fraud
xgb = XGBClassifier(scale_pos_weight=scale_weight, eval_metric='aucpr', random_state=42)

# Set up Stratified 5-Fold Cross Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# The metrics we want to track
scoring = {'pr_auc': 'average_precision', 'precision': 'precision', 'recall': 'recall'}

def run_cv(model, X, y, name):
    print(f"Running 5-fold CV for {name}...")
    # Run the 5 iterations (n_jobs=2 to speed it up safely)
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=2)
    
    return {
        'Model': name,
        'PR-AUC (Mean)': np.mean(scores['test_pr_auc']),
        'PR-AUC (Std)': np.std(scores['test_pr_auc']),
        'Precision (Mean)': np.mean(scores['test_precision']),
        'Precision (Std)': np.std(scores['test_precision']),
        'Recall (Mean)': np.mean(scores['test_recall']),
        'Recall (Std)': np.std(scores['test_recall']),
    }

## 2. Run Cross-Validation with Full Features

In [3]:
print("=== CV with ALL FEATURES ===")
results_full = []
results_full.append(run_cv(rf, X_full, y, 'Random Forest (Full)'))
results_full.append(run_cv(xgb, X_full, y, 'XGBoost (Full)'))

df_full = pd.DataFrame(results_full).set_index('Model')
display(df_full)

=== CV with ALL FEATURES ===
Running 5-fold CV for Random Forest (Full)...
Running 5-fold CV for XGBoost (Full)...


,PR-AUC (Mean),PR-AUC (Std),Precision (Mean),Precision (Std),Recall (Mean),Recall (Std)
Model,,,,,,
Random Forest (Full),0.998311,0.003143,0.996935,0.003754,0.996899,0.006202
XGBoost (Full),0.991948,0.004340,0.925541,0.016685,0.975242,0.010309


## 3. The Ablation Test
**What is an Ablation Test?**
In medicine, ablation means removing a part of the body to see if the disease goes away. In machine learning, it means removing specific features to see if the model's performance collapses. 

We are going to remove our handcrafted `errorBalanceOrig` and `errorBalanceDest` features. If the model's performance drops significantly, it proves to the interviewer that our feature engineering work was the critical key to solving the problem.

In [4]:
print("\n=== CV WITHOUT errorBalance FEATURES (Ablation Test) ===")
# Remove the engineered features
X_ablated = X_full.drop(columns=['errorBalanceOrig', 'errorBalanceDest'])

results_ablated = []
results_ablated.append(run_cv(rf, X_ablated, y, 'Random Forest (Ablated)'))
results_ablated.append(run_cv(xgb, X_ablated, y, 'XGBoost (Ablated)'))

df_ablated = pd.DataFrame(results_ablated).set_index('Model')
display(df_ablated)


=== CV WITHOUT errorBalance FEATURES (Ablation Test) ===
Running 5-fold CV for Random Forest (Ablated)...
Running 5-fold CV for XGBoost (Ablated)...


,PR-AUC (Mean),PR-AUC (Std),Precision (Mean),Precision (Std),Recall (Mean),Recall (Std)
Model,,,,,,
Random Forest (Ablated),0.894766,0.014681,0.956658,0.014717,0.806762,0.029235
XGBoost (Ablated),0.921638,0.011690,0.801934,0.030459,0.865546,0.026991


## 4. Side-by-Side Comparison

In [5]:
combined_df = pd.concat([df_full, df_ablated])

print("\n--- FINAL COMPARISON ---")
# Highlighting the max values in green to easily spot the winners
display(combined_df.style.highlight_max(subset=['PR-AUC (Mean)', 'Precision (Mean)', 'Recall (Mean)'], color='lightgreen', axis=0))


--- FINAL COMPARISON ---


,PR-AUC (Mean),PR-AUC (Std),Precision (Mean),Precision (Std),Recall (Mean),Recall (Std)
Model,,,,,,
Random Forest (Full),0.998311,0.003143,0.996935,0.003754,0.996899,0.006202
XGBoost (Full),0.991948,0.004340,0.925541,0.016685,0.975242,0.010309
Random Forest (Ablated),0.894766,0.014681,0.956658,0.014717,0.806762,0.029235
XGBoost (Ablated),0.921638,0.011690,0.801934,0.030459,0.865546,0.026991
